# 00 – Business Understanding

First phase of **CRISP-DM (Business Understanding)**. This notebook describes the domain, the client's requirements and how they become data mining goals. It follows the three Business Understanding criteria of the grading rubric:

1. **Analysis of requirements with the end user** (Section 1)
2. **Definition of business goals** (Section 2)
3. **Translation of business goals into data mining goals** (Section 3)

Business Understanding is mostly text. The few code cells only **verify domain facts against the data** (league structure, schedule, points rule, playoff format), so every statement about the domain is backed by the database built with `src/load_db.py`.

| Phase | Notebook |
|---|---|
| 1. Business Understanding | `00_business_understanding.ipynb` (this notebook) |
| 2. Data Understanding | `01_eda.ipynb`, `02_statistical_analysis.ipynb` |
| 3. Data Preparation | `03_data_preparation.ipynb` |

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../data/hockey.db")
q = lambda sql: pd.read_sql(sql, con)

## 1. Analysis of requirements with the end user

### 1.1 The game
- A game is played on an ice rink between two teams.
- Each team has six players on the ice at a time, including the goaltender.
- A game lasts one hour of gameplay (three periods of 20 minutes).
- The team that scores the most goals wins. Until 2003/04 a game could end in a tie; since 2005/06 ties are resolved by overtime and, if needed, a shootout.
- Players who break the rules are sent to the penalty box for a number of **penalty minutes (PIM)**: 2 for a minor penalty, 5 for a major (e.g. fighting), 10 for a misconduct.

### 1.2 The season
- Teams play a **regular season** inside a league divided into two conferences (Eastern and Western), each split into divisions. Teams accumulate points to reach the best possible standing.
- At the end of the regular season, the best teams qualify for the **playoffs**, where they play elimination series for the Stanley Cup.

### 1.3 League structure (last season in the data, 2009/10)

30 teams → 2 conferences of 15 → 3 geographic divisions of 5 per conference.

| Conference | Division | Teams (2009/10) |
|---|---|---|
| Eastern | Atlantic (AT) | New Jersey, Pittsburgh, Philadelphia, NY Rangers, NY Islanders |
| Eastern | Northeast (NE) | Boston, Buffalo, Ottawa, Montreal, Toronto |
| Eastern | Southeast (SE) | Washington, Carolina, Atlanta, Florida, Tampa Bay |
| Western | Central (CE) | Chicago, Detroit, St. Louis, Nashville, Columbus |
| Western | Northwest (NW) | Vancouver, Calgary, Minnesota, Colorado, Edmonton |
| Western | Pacific (PC) | San Jose, Phoenix, Los Angeles, Anaheim, Dallas |

Check in the data:

In [2]:
structure = q('''
SELECT confID, divID, COUNT(*) AS n_teams, GROUP_CONCAT(name, ', ') AS teams
FROM Teams
WHERE lgID = 'NHL' AND year = 2009
GROUP BY confID, divID
ORDER BY confID, divID
''')
pd.set_option("display.max_colwidth", None)
structure

,confID,divID,n_teams,teams
0,EC,AT,5,"New Jersey Devils, New York Islanders, New York Rangers, Philadelphia Flyers, Pittsburgh Penguins"
1,EC,NE,5,"Boston Bruins, Buffalo Sabres, Montreal Canadiens, Ottawa Senators, Toronto Maple Leafs"
2,EC,SE,5,"Atlanta Thrashers, Carolina Hurricanes, Florida Panthers, Tampa Bay Lightning, Washington Capitals"
3,WC,CE,5,"Columbus Blue Jackets, Chicago Blackhawks, Detroit Red Wings, Nashville Predators, St. Louis Blues"
4,WC,NW,5,"Calgary Flames, Colorado Avalanche, Edmonton Oilers, Minnesota Wild, Vancouver Canucks"
5,WC,PC,5,"Anaheim Ducks, Dallas Stars, Los Angeles Kings, Phoenix Coyotes, San Jose Sharks"


### 1.4 Regular-season schedule (82 games per team)

Every team plays every other team, but not the same number of times:

| Opponent | Number of opponents | Games vs each | Total |
|---|---|---|---|
| Same division | 4 | 6 | 24 |
| Same conference, other division | 10 | 4 | 40 |
| Other conference | 15 | 1 or 2 (2 against 3 of them, by rotation) | 18 |
| | | | **82** |

- About 78% of the games (64 of 82) are inside the conference, and 24 are against division rivals, so **the strength of a team's division affects its points** (strength of schedule).
- This format holds from 2008/09 to 2010/11. Earlier seasons had even fewer inter-conference games, and in some seasons not every pair of teams met.

Check in `TeamVsTeam` for 2009:

In [3]:
schedule = q('''
SELECT v.tmID, v.oppID,
       v.W + v.L + COALESCE(v.T, 0) + COALESCE(v.OTL, 0) AS games,
       CASE WHEN a.divID = b.divID THEN '1. Same division'
            WHEN a.confID = b.confID THEN '2. Same conference, other division'
            ELSE '3. Other conference' END AS relation
FROM TeamVsTeam v
JOIN Teams a ON a.year = v.year AND a.lgID = v.lgID AND a.tmID = v.tmID
JOIN Teams b ON b.year = v.year AND b.lgID = v.lgID AND b.tmID = v.oppID
WHERE v.lgID = 'NHL' AND v.year = 2009
''')
per_team = schedule.groupby(["tmID", "relation"]).agg(opponents=("oppID", "nunique"), games=("games", "sum"))
summary = per_team.groupby("relation").agg(
    opponents_per_team=("opponents", "mean"),
    games_per_team=("games", "mean"),
)
summary["games_vs_each"] = summary["games_per_team"] / summary["opponents_per_team"]
print(f"Games per team in 2009 (sum over opponents): {schedule.groupby('tmID')['games'].sum().unique()}")
summary.round(2)

Games per team in 2009 (sum over opponents): [82.]


,opponents_per_team,games_per_team,games_vs_each
relation,,,
1. Same division,4.0,24.0,6.0
"2. Same conference, other division",10.0,40.0,4.0
3. Other conference,15.0,18.0,1.2


### 1.5 Points and standings

| Result | Points |
|---|---|
| Win (regulation, overtime or shootout) | 2 |
| Overtime / shootout loss (OTL, since 1999/00) | 1 |
| Tie (only until 2003/04) | 1 |
| Regulation loss | 0 |

- $\text{Pts} = 2 \cdot W + T + \text{OTL}$. Since 2005/06 there are no ties: 5 minutes of overtime, then a shootout.
- Seasons have different lengths in 1982–2009 (80 games until 1991/92, 84 in 1992/93 and 1993/94, 48 in the 1994/95 lockout, 82 from 1995/96, and no 2004/05 season at all), so teams are compared with **points% = Pts / (2·G)**.
- Three standings exist at the same time, all sorted by points: division (5 teams; the `rank` column of `Teams`), conference (15 teams; decides the playoffs) and league (30 teams; the best team wins the Presidents' Trophy).
- Tie-break on equal points: more wins first (the official rule then uses head-to-head results and goal differential).
- **Conference standing:** the 3 division winners take places 1–3 of the conference, even with fewer points than other teams; the other 12 teams follow by points from 4th place.

Check of the points rule and of the season lengths:

In [4]:
pts_check = q('''
SELECT COUNT(*) AS nhl_team_seasons_1982_2009,
       SUM(CASE WHEN Pts = 2 * W + COALESCE(T, 0) + COALESCE(OTL, 0) THEN 1 ELSE 0 END) AS rule_holds
FROM Teams
WHERE lgID = 'NHL' AND year >= 1982
''')
display(pts_check)

season_length = q('''
SELECT year, COUNT(*) AS teams, MIN(G) AS games_min, MAX(G) AS games_max
FROM Teams
WHERE lgID = 'NHL' AND year >= 1982
GROUP BY year
''')
season_length.groupby(["games_min", "games_max"])["year"].agg(["min", "max", "count"]).rename(
    columns={"min": "first_season", "max": "last_season", "count": "n_seasons"})

,nhl_team_seasons_1982_2009,rule_holds
0,690,690


,,first_season,last_season,n_seasons
games_min,games_max,,,
48,48,1994,1994,1
80,80,1982,1991,10
82,82,1995,2009,14
84,84,1992,1993,2


### 1.6 Playoffs
- The top 8 of each conference qualify (16 teams).
- Round 1 is played inside each conference: 1st vs 8th, 2nd vs 7th, 3rd vs 6th, 4th vs 5th. The following rounds are also inside the conference.
- The two conference champions play the Stanley Cup final. Every series is best of 7 (first to 4 wins).

Check: how many teams played the playoffs in 2009, per conference?

In [5]:
q('''
SELECT t.confID, COUNT(*) AS playoff_teams
FROM TeamsPost p
JOIN Teams t ON t.year = p.year AND t.lgID = p.lgID AND t.tmID = p.tmID
WHERE p.lgID = 'NHL' AND p.year = 2009
GROUP BY t.confID
''')

,confID,playoff_teams
0,EC,8
1,WC,8


### 1.7 Requirements

Using the provided historical data, predict for the target season:

- **(a)** The final regular-season ranking of the teams.
- **(b)** The set of teams that will change coaches. Following the professor's clarification, a coaching change counts when it happens **during** the season, i.e. whether the team replaces its coach during the current season.
- **(c)** How many penalty minutes (PIM) each player will accumulate during the regular season. This requirement was announced later in the project; the solution was kept modular so that it could be added with little rework.

### 1.8 End users and objectives

| Requirement | End users | How they would use the prediction |
|---|---|---|
| (a) Final ranking | Scouts, bettors, analysts, media | Scouting and roster planning, betting decisions, forecasting which teams reach the playoffs |
| (b) Coach change | Team management, media, contract and staffing planning | Management decisions, contract planning, anticipating the coaching market, media coverage |
| (c) Penalty minutes | *TODO* | *TODO* |

> **TODO (team):** fill in the end users of (c) and how they would use the prediction, then delete this note.

### 1.9 What each user needs from the output
- **(a)** A complete ranking of all teams. Users benefit from knowing which teams are likely to land near the playoff cut-off.
- **(b)** A list of teams predicted to change coach, with a probability or risk score per team.
- **(c)** *TODO (team)*

### 1.10 Constraints and assumptions
- **Prediction time:** predictions are made the day before the first game of the new season. Only information available at that moment may be used.
- **Data:** only the historical data provided in the assignment is used.
- **Coaching change timing:** a change counts when it happens during the season (see 1.7).
- **Penalty minutes:** only regular-season PIM are predicted (playoff PIM are excluded), one value per player and season. A player traded during the season has one row per team in the data; his PIM are added up.

### 1.11 Expectations on quality
- **(a)** The predicted ranking should be close to the real final standings, and especially reliable at separating playoff teams from non-playoff teams.
- **(b)** The model should identify most of the teams that change coach without producing too many false alarms.
- **(c)** *TODO (team)*

### 1.12 Open points
- **Division-winner rule:** the conference ranking must put the predicted division winners in places 1–3 (see 1.5). It hardly changes who makes the playoffs, but it changes the order inside the top 8. To apply when the ranking of task (a) is built.
- **Strength of schedule:** 24 of the 82 games are against division rivals (see 1.4). This became the feature `div_rivals_pts_pct_rel` (mean relative points% of the division rivals), built in Data Understanding (`sql/playoff_and_sos.sql`).

## 2. Definition of business goals

| Business goal |
|---|
| I want to know, before the season starts, how teams will finish. |
| Will the team make the playoffs? |
| I want to know which teams will change coach. |
| *TODO (team): business goal of (c)* |

## 3. Translation of business goals into data mining goals

| Business goal | Data mining goal | Type |
|---|---|---|
| I want to know, before the season starts, how teams will finish. | Predict each team's points percentage Pts / (2·G) for season $t$, then sort the teams (within division, conference and league) | Regression, then ranking |
| Will the team make the playoffs? | Predict playoff yes/no | Binary classification |
| I want to know which teams will change coach. | Predict whether the team changes coach during season $t$ | Binary classification |
| *TODO (team): business goal of (c)* | Predict each player's regular-season PIM for season $t$ | Regression (count target) |

> **TODO (team):** add, for each data mining goal, the evaluation metric and the success criterion (e.g. "beat the *same as last season* baseline"), then delete this note.

In [6]:
con.close()